# Kaggle Invoice Dataset - Download + Train
Notebook recable sur le dataset officiel Kaggle: `osamahosamabdellatif/high-quality-invoice-images-for-ocr`.


## Cible ML explicite (etat actuel)
- Le pipeline entraine un **classifieur d'images** sur le dataset Kaggle (pas d'OCR complet).
- Les labels sont resolus depuis les CSV (`label/class/...`) quand disponibles.
- Si un label CSV est absent, fallback explicite: **nom du dossier parent de l'image** (ex: `batch1_1`).
- Cette politique est exposee dans le `summary` JSON (`label_policy`, `label_source_counts`).


In [1]:
# Optionnel si kagglehub n'est pas installe
# !pip install kagglehub


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("osamahosamabdellatif/high-quality-invoice-images-for-ocr")

print("Path to dataset files:", path)


D:\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Francis BACKELAND\.cache\kagglehub\datasets\osamahosamabdellatif\high-quality-invoice-images-for-ocr\versions\3


In [3]:
from pathlib import Path

DATA_ROOT = Path(path).resolve()
batch_dirs = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith('batch_')])
csv_files = sorted(DATA_ROOT.rglob('*.csv'))
print('DATA_ROOT =', DATA_ROOT)
print('batch dirs =', batch_dirs[:10])
print('num csv =', len(csv_files))
if not batch_dirs:
    raise RuntimeError('Structure attendue absente: batch_*')
if not csv_files:
    raise RuntimeError('Aucun CSV trouve dans le dataset.')


DATA_ROOT = C:\Users\Francis BACKELAND\.cache\kagglehub\datasets\osamahosamabdellatif\high-quality-invoice-images-for-ocr\versions\3
batch dirs = ['batch_1', 'batch_2', 'batch_3']
num csv = 3


In [4]:
import json
from src.scanned_images_pipeline import PipelineConfig, run_training_pipeline

config = PipelineConfig(
    project_root=Path.cwd().resolve(),
    data_root=DATA_ROOT,
    artifacts_dir=Path.cwd().resolve() / 'artifacts',
    epochs=1,
    batch_size=8,
    img_size=224,
    use_pretrained=False,
)
summary = run_training_pipeline(config)
print(json.dumps({
    'data_root': summary['data_root'],
    'num_manifest_records': summary['num_manifest_records'],
    'label_policy': summary['label_policy'],
    'label_source_counts': summary['label_source_counts'],
    'num_train_samples': summary['num_train_samples'],
    'num_val_samples': summary['num_val_samples'],
    'model_path': summary['model_path'],
}, indent=2))


{
  "data_root": "C:\\Users\\Francis BACKELAND\\.cache\\kagglehub\\datasets\\osamahosamabdellatif\\high-quality-invoice-images-for-ocr\\versions\\3",
  "num_manifest_records": 1414,
  "label_policy": "Prefer CSV label/class columns; fallback to image parent folder name when label is missing.",
  "label_source_counts": {
    "parent_folder_fallback": 1414
  },
  "num_train_samples": 1131,
  "num_val_samples": 283,
  "model_path": "D:\\Projet\\scanned_images_classifier\\notebooks\\artifacts\\scanned_images_resnet18.pt"
}
